# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. [1. 2. 3.] и [0 1 1]. Потому что этот оператор / возвращает результат с float, а оператор // возвращает результат с округлением в int
2. [4. 4. 4.]. Потому что np.ones выдает результат с float, а также axis=1 суммирует числа по каждой строке
3. [[0 1 2]
  [1 2 3]
  [2 3 4]] Потому что создается два листа где один просто [0,1,2] а второй такой же но вертикальный из за reshape, и потом строка [0,1,2] складывается к каждому значению столбца
4. 6.0. Потому что sqrt вычисляет квадратный корень каждого элемента, а потом sum суммирует все элементы
5. 7.5. Потому что **2 вычисляет квадрат каждого элемента и затем вычисляет среднее значение

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


In [20]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [21]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "/ в чистых данных:", clean.sum())

строк всего: 5463 / в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [38]:
# 1
month = np.array([r["order_date"][:7] for r in rows])

september = amount[clean & (month == "2026-09")].sum()
october = amount[clean & (month == "2026-10")].sum()
change = (october - september) / september * 100

print(f"Строк осталось: {clean.sum()}")
print(f"Выручка сентября: {september:,.2f} тг")
print(f"Выручка октября: {october:,.2f} тг")
print(f"Изменение: {change:+.2f}%")

Строк осталось: 4739
Выручка сентября: 35,243,410.00 тг
Выручка октября: 34,419,520.00 тг
Изменение: -2.34%


С ДЗ 2 не совпадает, там мы учитывали все оплаченные заказы включая дубликаты и test

## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [23]:
# 2
categories = np.unique(cat)
months = ["2026-09", "2026-10"]

table = np.zeros((len(categories), 2))
for i in range(len(categories)):
    for j in range(len(months)):
        mask = clean & (cat == categories[i]) & (month == months[j])
        table[i, j] = amount[mask].sum()

print(f"{'Категория':<15} {'Сентябрь':>15} {'Октябрь':>15}")
for i in range(len(categories)):
    print(f"{categories[i]:<15} {table[i, 0]:>15.2f} {table[i, 1]:>15.2f}")

Категория              Сентябрь         Октябрь
Дом                  4997730.00      5125790.00
Книги                1065300.00      1140910.00
Красота              2610950.00      2678990.00
Одежда               9167100.00      8505730.00
Электроника         17402330.00     16968100.00


In [36]:
total = table.sum(axis=1)
leader = total.argmax()

print(f"Больше всего выручки: {categories[leader]} - {total[leader]:.2f} тг")
change = (table[:, 1] - table[:, 0]) / table[:, 0] * 100
worst = change.argmin()
print(f"Сильнее всего упала: {categories[worst]} - {change[worst]:+.2f}%")

Больше всего выручки: Электроника - 34370430.00 тг
Сильнее всего упала: Одежда - -7.21%


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [25]:
# 3
counts = np.zeros((len(categories), 2))

for i in range(len(categories)):
    for j in range(len(months)):
        mask = clean & (cat == categories[i]) & (month == months[j])
        counts[i, j] = mask.sum()
average = table / counts

print(f"{'Категория':<15} {'Сентябрь':>15} {'Октябрь':>15}")

for i in range(len(categories)):
    print(f"{categories[i]:<15} {average[i, 0]:>15.2f} {average[i, 1]:>15.2f}")

Категория              Сентябрь         Октябрь
Дом                    10499.43        10768.47
Книги                   4020.00         3815.75
Красота                 6187.09         6158.60
Одежда                 13520.80        13045.60
Электроника            32772.75        33600.20


In [35]:
overall_average = table.sum(axis=1) / counts.sum(axis=1)
best = overall_average.argmax()

print(f"Самый высокий средний чек: {categories[best]} - {overall_average[best]:.2f} тг")

Самый высокий средний чек: Электроника - 33176.09 тг


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [31]:
# 4
cities = np.unique(city)
months = ["2026-09", "2026-10"]
month = np.array([r["order_date"][:7] for r in rows])

city_table = np.zeros((len(cities), 2))

for i in range(len(cities)):
    for j in range(len(months)):
        mask = clean & (city == cities[i]) & (month == months[j])
        city_table[i, j] = amount[mask].sum()

shares = city_table / city_table.sum(axis=0)
print(f"{'Город':<15} {'Сентябрь':>12} {'Октябрь':>12}")
for i in range(len(cities)):
    print(f"{cities[i]:<15} {shares[i, 0]:>12.2%} {shares[i, 1]:>12.2%}")

print("\nСуммы столбцов:", shares.sum(axis=0))
print("Проверка:", np.allclose(shares.sum(axis=0), 1))

shymkent = shares[cities == "Шымкент"][0]
print(f"\nДоля Шымкента в сентябре: {shymkent[0]:.2%}")
print(f"Доля Шымкента в октябре: {shymkent[1]:.2%}")

difference = (shymkent[1] - shymkent[0]) * 100
print(f"Изменение доли: {difference:+.2f}")

Город               Сентябрь      Октябрь
Актобе                 7.31%        7.38%
Алматы                41.15%       42.54%
Астана                25.63%       25.71%
Атырау                 2.62%        2.82%
Караганда             10.35%        9.80%
Шымкент               12.93%       11.75%

Суммы столбцов: [1. 1.]
Проверка: True

Доля Шымкента в сентябре: 12.93%
Доля Шымкента в октябре: 11.75%
Изменение доли: -1.19


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [34]:
# 5
change = city_table[:, 1] - city_table[:, 0]
contribution = change / change.sum()

for i in range(len(cities)):
    print(f"{cities[i]}: {contribution[i]:.1%}")

shymkent = contribution[cities == "Шымкент"][0]
print(f"\nШымкент даёт {shymkent:.1%} общего падения")

print(
    f"Менеджеру: Шымкент даёт {shymkent:.1%} общего падения выручки - "
    "начните с проверки изменения числа оплаченных заказов "
    "и среднего чека в этом городе"
)

Актобе: 4.4%
Алматы: -17.2%
Астана: 22.3%
Атырау: -5.6%
Караганда: 33.6%
Шымкент: 62.5%

Шымкент даёт 62.5% общего падения
Менеджеру: Шымкент даёт 62.5% общего падения выручки - начните с проверки изменения числа оплаченных заказов и среднего чека в этом городе


## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [32]:
# 6
share = table / table.sum(axis=1, keepdims=True)

У таблицы городов форма (6, 2), а table.sum(axis=1) возвращает массив формы (6,), тоесть общую выручку каждого города. И из за того что размеры массивов не сопостовляются возникает ошибка. Нужно сохранить суммы столбца в форме (6, 1)

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [ ]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8